# Notebook 04: Evaluation & VIRVS Benchmarking Suite

### Overview
In this notebook, we perform quantitative benchmarking comparing the **U-Net Baseline** and the **Pix2Pix Conditional GAN** models on the VIRVS test set.

**Key Learning Objectives**:
1. Calculate standard image fidelity metrics: **PSNR**, **SSIM**, **PCC**, and **MAE**.
2. Generate a structured quantitative benchmark comparison table.
3. Perform single-cell infection reporter signal quantification ($I_{\text{viral}}$).
4. Plot ground truth vs predicted viral reporter intensities across cells.

In [ ]:
# ==========================================================
# 1. Google Colab Setup & Environment Setup
# ==========================================================
import sys
import os

if 'google.colab' in sys.modules:
    print("[+] Google Colab detected!")
    !git clone https://github.com/ayakimovich/GenAI4BIA.git /content/GenAI4BIA
    %cd /content/GenAI4BIA/practical
    !pip install -r ../requirements.txt -q
    sys.path.append(os.path.abspath("."))
else:
    sys.path.append(os.path.abspath("."))

## 2. Load Checkpoints & Run Validation Inference

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

from src.data import VIRVSDataset
from src.models import UNet, Pix2PixGenerator
from src.metrics import compute_psnr, compute_ssim, compute_pcc, compute_mae, compute_cell_reporter_stats
from src.generate_mock_virvs_data import create_dataset_directory, generate_virvs_pair
from src.utils import plot_virtual_staining_comparison

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

data_dir = "./data/mock_virvs"
create_dataset_directory(data_dir, num_train=30, num_val=10, image_size=(256, 256))

# Load validation set in [0, 1] for metric calculation
val_dataset = VIRVSDataset(root_dir=data_dir, split="val", normalize_range=(0.0, 1.0))
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False)

# Load U-Net & Pix2Pix Checkpoints
unet = UNet(in_channels=1, out_channels=1, features=[32, 64, 128, 256]).to(device)
if os.path.exists("./unet_virvs_baseline.pth"):
    unet.load_state_dict(torch.load("./unet_virvs_baseline.pth", map_location=device))
unet.eval()

pix2pix = Pix2PixGenerator(in_channels=1, out_channels=1, features=[32, 64, 128, 256]).to(device)
if os.path.exists("./pix2pix_generator_virvs.pth"):
    pix2pix.load_state_dict(torch.load("./pix2pix_generator_virvs.pth", map_location=device))
pix2pix.eval()

## 3. Quantitative Image Fidelity Metrics (PSNR, SSIM, PCC, MAE)

In [ ]:
metrics_unet = {"psnr": [], "ssim": [], "pcc": [], "mae": []}
metrics_pix2pix = {"psnr": [], "ssim": [], "pcc": [], "mae": []}

with torch.no_grad():
    for batch in val_loader:
        bf = batch["brightfield"].to(device)       # [1, 1, H, W] in [0, 1]
        target = batch["fluorescence"].to(device)   # [1, 1, H, W] in [0, 1]
        
        # U-Net prediction in [0, 1]
        pred_unet = unet(bf)
        
        # Pix2Pix prediction (convert input to [-1, 1], convert output back to [0, 1])
        bf_pix = bf * 2.0 - 1.0
        pred_pix_raw = pix2pix(bf_pix)
        pred_pix = (pred_pix_raw + 1.0) / 2.0
        
        # Record U-Net metrics
        metrics_unet["psnr"].append(compute_psnr(target, pred_unet))
        metrics_unet["ssim"].append(compute_ssim(target, pred_unet))
        metrics_unet["pcc"].append(compute_pcc(target, pred_unet))
        metrics_unet["mae"].append(compute_mae(target, pred_unet))
        
        # Record Pix2Pix metrics
        metrics_pix2pix["psnr"].append(compute_psnr(target, pred_pix))
        metrics_pix2pix["ssim"].append(compute_ssim(target, pred_pix))
        metrics_pix2pix["pcc"].append(compute_pcc(target, pred_pix))
        metrics_pix2pix["mae"].append(compute_mae(target, pred_pix))

# Create Benchmark Summary Table
summary_data = {
    "Model": ["U-Net Baseline", "Pix2Pix cGAN"],
    "PSNR (dB) ↑": [f"{np.mean(metrics_unet['psnr']):.2f} ± {np.std(metrics_unet['psnr']):.2f}",
                    f"{np.mean(metrics_pix2pix['psnr']):.2f} ± {np.std(metrics_pix2pix['psnr']):.2f}"],
    "SSIM ↑": [f"{np.mean(metrics_unet['ssim']):.3f} ± {np.std(metrics_unet['ssim']):.3f}",
               f"{np.mean(metrics_pix2pix['ssim']):.3f} ± {np.std(metrics_pix2pix['ssim']):.3f}"],
    "PCC ↑": [f"{np.mean(metrics_unet['pcc']):.3f} ± {np.std(metrics_unet['pcc']):.3f}",
              f"{np.mean(metrics_pix2pix['pcc']):.3f} ± {np.std(metrics_pix2pix['pcc']):.3f}"],
    "MAE ↓": [f"{np.mean(metrics_unet['mae']):.4f} ± {np.std(metrics_unet['mae']):.4f}",
              f"{np.mean(metrics_pix2pix['mae']):.4f} ± {np.std(metrics_pix2pix['mae']):.4f}"]
}

df_results = pd.DataFrame(summary_data)
print("\n================ VIRVS BENCHMARK SUMMARY ================")
print(df_results.to_string(index=False))

## 4. Single-Cell Infection Reporter Signal Quantification
We evaluate how well predicted virtual stains quantify true single-cell viral infection intensity.

In [ ]:
# Generate a test sample with known single-cell masks
bf_arr, fluo_arr, cell_masks = generate_virvs_pair(image_size=(256, 256), seed=999)

bf_t = torch.from_numpy(bf_arr).unsqueeze(0).unsqueeze(0).to(device)
fluo_t = torch.from_numpy(fluo_arr).unsqueeze(0).unsqueeze(0).to(device)

with torch.no_grad():
    pred_unet_cell = unet(bf_t)[0, 0].cpu().numpy()
    pred_pix_cell = ((pix2pix(bf_t * 2.0 - 1.0)[0, 0] + 1.0) / 2.0).cpu().numpy()

# Compute single-cell viral reporter intensity stats
unet_cell_stats = compute_cell_reporter_stats(fluo_arr, pred_unet_cell, cell_masks)
pix_cell_stats = compute_cell_reporter_stats(fluo_arr, pred_pix_cell, cell_masks)

plt.figure(figsize=(7, 5))
plt.scatter(unet_cell_stats["true_intensities"], unet_cell_stats["pred_intensities"], 
            color="tab:blue", label=f"U-Net (Cell PCC: {unet_cell_stats['cell_pcc']:.2f})", s=50, alpha=0.8)
plt.scatter(pix_cell_stats["true_intensities"], pix_cell_stats["pred_intensities"], 
            color="tab:orange", label=f"Pix2Pix (Cell PCC: {pix_cell_stats['cell_pcc']:.2f})", s=50, alpha=0.8)
plt.plot([0, 1], [0, 1], "k--", label="Ideal 1:1 Line", alpha=0.7)

plt.xlabel("True Single-Cell Viral Reporter Intensity", fontweight="bold")
plt.ylabel("Predicted Virtual Stain Intensity", fontweight="bold")
plt.title("Single-Cell Infection Quantification Fidelity", fontweight="bold")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# Side-by-Side Visual Comparison
plot_virtual_staining_comparison(
    bf_arr, fluo_arr, pred_unet=pred_unet_cell, pred_pix2pix=pred_pix_cell,
    title="VIRVS Benchmark Comparison: U-Net vs Pix2Pix"
)